# 01 — Análise Exploratória de Dados (EDA)
## Credit Card Fraud Detection

---

### Objetivo deste notebook

Antes de construir qualquer modelo, precisamos **entender profundamente os dados**. A EDA responde perguntas fundamentais que guiam todas as decisões de modelagem:

| Pergunta | Impacto na modelagem |
|---|---|
| Quão desbalanceado é o dataset? | Define qual métrica usar (PR-AUC, não Acurácia) |
| Existem valores nulos? | Define estratégia de imputação |
| Como fraudes diferem de legítimas? | Valida a hipótese do Autoencoder |
| Qual a escala de Amount e Time? | Justifica o uso de StandardScaler |
| As features PCA têm distribuição normal? | Confirma que não precisam de transformação extra |

---

### Sobre o dataset

O dataset **Credit Card Fraud Detection** foi produzido pela Université Libre de Bruxelles (ULB) e contém transações reais de cartões de crédito europeus de setembro de 2013.

- **284.807 transações** em 2 dias
- **492 fraudes** — apenas **0,17%** do total
- Features `V1`–`V28`: componentes PCA aplicado sobre dados reais anonimizados por questões de privacidade
- `Time`: segundos decorridos desde a primeira transação no dataset
- `Amount`: valor da transação em euros
- `Class`: variável alvo — 0 = legítima, 1 = fraude

---
## 1. Setup e Imports

Carregamos as bibliotecas essenciais para análise e visualização.

- **numpy / pandas**: manipulação numérica e tabular dos dados
- **matplotlib / seaborn**: visualizações estáticas de alta qualidade
- **pathlib.Path**: manipulação de caminhos independente de sistema operacional (funciona igual no Windows e Linux — importante para reprodutibilidade)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurações visuais globais
# figsize padrão evita gráficos achatados
plt.rcParams['figure.figsize'] = (12, 6)
# Remove bordas superior e direita — visual mais limpo
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
# Paleta com boa distinção entre categorias
sns.set_palette('husl')

# Caminho relativo ao notebook
# notebooks/ está um nível abaixo da raiz do projeto
DATA_PATH = Path('../data/raw/creditcard.csv')

print('Setup concluido.')

---
## 2. Carregamento e Inspeção Inicial

A inspeção inicial responde: **os dados chegaram íntegros?**

Verificamos shape, tipos de dados e valores nulos. Problemas aqui invalidariam toda a modelagem.

In [ ]:
# Carrega o CSV — pandas infere tipos automaticamente
df = pd.read_csv(DATA_PATH)

print(f'Shape   : {df.shape}  ->  {df.shape[0]:,} linhas x {df.shape[1]} colunas')
print(f'Memoria : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print(f'\nColunas ({df.shape[1]}):')
print(list(df.columns))
df.head(3)

In [ ]:
# Estatísticas descritivas das features nao anonimizadas
# V1-V28 nao sao exibidas aqui pois ja foram normalizadas pelo PCA
print('Estatisticas de Time, Amount e Class:')
df[['Time', 'Amount', 'Class']].describe().round(4)

In [ ]:
# Verificacao de valores nulos
# Nulos exigiriam estrategia de imputacao (media, mediana, KNN, etc.)
# No dataset ULB nao ha nulos, mas sempre verificamos antes de assumir
nulls = df.isnull().sum()
total_nulls = nulls.sum()

if total_nulls == 0:
    print('Nenhum valor nulo encontrado — dataset completo.')
else:
    print(f'{total_nulls} valores nulos encontrados:')
    print(nulls[nulls > 0])

---
## 3. Desbalanceamento de Classes

Esta é a característica mais crítica do dataset e a que mais impacta as decisões de modelagem.

### Por que o desbalanceamento é um problema?

Um classificador que **sempre prevê 'legítima'** obtém:
- **Acurácia: 99,83%** — parece excelente
- **Recall de fraude: 0%** — não detecta nenhuma fraude — inútil para o negócio

Isso explica por que **acurácia não é uma métrica válida** aqui, e por que usamos **PR-AUC** (Area Under the Precision-Recall Curve).

### Estratégias adotadas para lidar com o desbalanceamento:
1. **SMOTE**: gera amostras sintéticas de fraude no treino (eleva para ~10% da classe)
2. **`scale_pos_weight`** no XGBoost: penaliza erros na classe minoritária
3. **Threshold calibrado**: não usamos 0,5 — encontramos o ponto que maximiza F1 na curva PR
4. **Autoencoder**: abordagem unsupervised que não depende de labels balanceados

In [ ]:
# Contagem e proporcao de cada classe
class_counts = df['Class'].value_counts().sort_index()
fraud_pct = df['Class'].mean() * 100
legit_pct = 100 - fraud_pct
ratio = int(class_counts[0] / class_counts[1])

print('=' * 50)
print(f'  Total de transacoes  : {len(df):>10,}')
print(f'  Legitimas (Class=0)  : {class_counts[0]:>10,}  ({legit_pct:.4f}%)')
print(f'  Fraudes   (Class=1)  : {class_counts[1]:>10,}  ({fraud_pct:.4f}%)')
print(f'  Proporcao            :   1 fraude : {ratio:,} legitimas')
print('=' * 50)
print()
print('Modelo naive que sempre prevê "legitima":')
print(f'  Acuracia = {legit_pct:.2f}%  (enganosamente alta)')
print(f'  Recall   = 0.00%    (detecta ZERO fraudes)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuicao das Classes — Desbalanceamento Extremo', fontsize=15, fontweight='bold', y=1.02)

colors = ['#27ae60', '#e74c3c']

# --- Grafico 1: Barras com escala logaritmica ---
# Usamos escala log no eixo Y porque sem ela a barra de fraude (492)
# ficaria invisivel ao lado da barra de legitimas (284.315)
bars = axes[0].bar(['Legitima (0)', 'Fraude (1)'], class_counts.values,
                   color=colors, edgecolor='white', linewidth=2, width=0.5)
axes[0].set_yscale('log')
axes[0].set_title('Contagem por Classe (escala log)', fontweight='bold')
axes[0].set_ylabel('Quantidade (escala logaritmica)')
for bar, count, pct in zip(bars, class_counts.values, [legit_pct, fraud_pct]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.8,
                 f'{count:,}\n({pct:.2f}%)', ha='center', va='bottom',
                 fontweight='bold', fontsize=10)

# --- Grafico 2: Pizza ---
# O explode (afastamento da fatia) evidencia visualmente o quanto
# a classe de fraude é minuscula na proporcao real
explode = (0, 0.15)
axes[1].pie(class_counts.values,
            labels=[f'Legitima\n{legit_pct:.2f}%', f'Fraude\n{fraud_pct:.4f}%'],
            colors=colors, explode=explode,
            autopct='%1.3f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2},
            textprops={'fontsize': 11})
axes[1].set_title('Proporcao Real das Classes', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Análise de Amount e Time

Estas são as únicas features **não anonimizadas** no dataset.

### Por que aplicar StandardScaler apenas em Amount e Time?

As features `V1`–`V28` **já foram normalizadas** pelo PCA (média ≈ 0, desvio padrão ≈ 1). `Amount` e `Time` têm escalas completamente diferentes:
- `Amount`: varia de 0 a ~25.000 (assimétrico à direita)
- `Time`: varia de 0 a ~172.000 (segundos)

Sem normalização, o XGBoost lidaria bem (é invariante à escala), mas o Autoencoder — baseado em MSE — penalizaria desproporcionalmente features com valores maiores.

In [ ]:
# Separamos as classes para comparar distribuicoes
legit = df[df['Class'] == 0]
fraud = df[df['Class'] == 1]

print('Estatisticas de Amount por classe:')
print(df.groupby('Class')['Amount'].describe().round(2).rename(index={0: 'Legitima', 1: 'Fraude'}))
print()
print('Estatisticas de Time por classe (em segundos):')
print(df.groupby('Class')['Time'].describe().round(0).rename(index={0: 'Legitima', 1: 'Fraude'}))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Analise de Amount e Time por Classe', fontsize=15, fontweight='bold')

# --- Amount: histograma comparativo ---
# density=True normaliza a area para 1, permitindo comparar distribuicoes
# de tamanhos muito diferentes (492 fraudes vs 284.315 legitimas)
# Sem density=True a barra de fraude seria invisivel
axes[0, 0].hist(legit['Amount'], bins=100, alpha=0.6, color='#27ae60',
                label=f'Legitima (n={len(legit):,})', density=True)
axes[0, 0].hist(fraud['Amount'], bins=50, alpha=0.8, color='#e74c3c',
                label=f'Fraude (n={len(fraud):,})', density=True)
axes[0, 0].set_title('Distribuicao do Amount', fontweight='bold')
axes[0, 0].set_xlabel('Amount (euros)')
axes[0, 0].set_ylabel('Densidade de probabilidade')
axes[0, 0].set_xlim(0, 500)
axes[0, 0].legend()

# --- Amount: boxplot ---
# Boxplot revela mediana, quartis e outliers
# notch=False: sem entalhes de IC (nao necessarios aqui)
# flierprops com alpha baixo: muitos outliers nao poluem o grafico
bp = axes[0, 1].boxplot([legit['Amount'], fraud['Amount']],
                         labels=['Legitima', 'Fraude'],
                         patch_artist=True,
                         medianprops=dict(color='#2c3e50', linewidth=2.5),
                         flierprops=dict(marker='o', markersize=2, alpha=0.3))
bp['boxes'][0].set_facecolor('#27ae6055')
bp['boxes'][1].set_facecolor('#e74c3c55')
axes[0, 1].set_title('Boxplot do Amount (limitado a 500 euros)', fontweight='bold')
axes[0, 1].set_ylabel('Amount (euros)')
axes[0, 1].set_ylim(0, 500)

# --- Time: distribuicao temporal ---
# Convertemos segundos para horas para facilitar interpretacao humana
# O padrao de 2 picos nas legitimas reflete 2 dias de dados
# (pico de movimento durante o horario comercial, queda a noite)
axes[1, 0].hist(legit['Time'] / 3600, bins=48, alpha=0.6, color='#27ae60',
                label='Legitima', density=True)
axes[1, 0].hist(fraud['Time'] / 3600, bins=48, alpha=0.8, color='#e74c3c',
                label='Fraude', density=True)
axes[1, 0].set_title('Distribuicao Temporal (horas)', fontweight='bold')
axes[1, 0].set_xlabel('Horas desde o inicio do dataset')
axes[1, 0].set_ylabel('Densidade')
axes[1, 0].legend()
axes[1, 0].axvline(24, color='gray', linestyle=':', alpha=0.7)
axes[1, 0].text(24.5, axes[1, 0].get_ylim()[1] * 0.85, 'Dia 2 ->', fontsize=9, color='gray')

# --- Amount: ECDF ---
# ECDF (Funcao de Distribuicao Acumulada Empirica):
# mostra a proporcao de transacoes ABAIXO de cada valor
# Ex: ECDF(100) = 0.8 significa que 80% das transacoes sao <= 100 euros
# Mais informativa que histograma para comparar distribuicoes
legit_sorted = np.sort(legit['Amount'].values)
fraud_sorted = np.sort(fraud['Amount'].values)
axes[1, 1].plot(legit_sorted, np.arange(1, len(legit_sorted)+1) / len(legit_sorted),
                color='#27ae60', label='Legitima', linewidth=2)
axes[1, 1].plot(fraud_sorted, np.arange(1, len(fraud_sorted)+1) / len(fraud_sorted),
                color='#e74c3c', label='Fraude', linewidth=2)
axes[1, 1].set_title('ECDF do Amount', fontweight='bold')
axes[1, 1].set_xlabel('Amount (euros)')
axes[1, 1].set_ylabel('Proporcao acumulada')
axes[1, 1].set_xlim(0, 500)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Análise das Features PCA (V1–V28)

As features `V1`–`V28` são componentes principais de PCA sobre dados originais anonimizados. O PCA por construção produz componentes:
- **Ortogonais** (descorrelacionados entre si)
- Com **variância decrescente**: V1 captura mais variância que V2, e assim por diante

### Por que isso valida o Autoencoder?

Se fraudes e legítimas têm **distribuições diferentes** nas features PCA, o Autoencoder — treinado apenas com legítimas — vai **falhar em reconstruir fraudes**, gerando alto `reconstruction_error`. Esse erro se torna o `anomaly score` que alimenta o XGBoost como feature extra.

In [ ]:
pca_cols = [f'V{i}' for i in range(1, 29)]

# Diferenca absoluta de medias entre classes por feature
# Quanto maior a diferenca, mais a feature distingue fraudes de legitimas
# Features com grande diferenca = maior poder discriminativo para o classificador
means_legit = df[df['Class']==0][pca_cols].mean()
means_fraud  = df[df['Class']==1][pca_cols].mean()
diff = (means_fraud - means_legit).abs().sort_values(ascending=False)

print('Top 10 features com maior diferenca de media entre classes:')
print(diff.head(10).round(4))
print()
print(f'Features com diferenca acima da mediana ({diff.median():.3f}): {(diff > diff.median()).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Analise das Features PCA (V1-V28)', fontsize=15, fontweight='bold')

# --- Diferenca de medias entre classes ---
# Destaca em vermelho as features acima da mediana de diferenca
# Sao as mais importantes para o classificador
colors_bar = ['#e74c3c' if d > diff.median() else '#95a5a6' for d in diff.values]
axes[0].bar(diff.index, diff.values, color=colors_bar, edgecolor='white')
axes[0].set_title('|Media Fraude - Media Legitima| por Feature', fontweight='bold')
axes[0].set_xlabel('Feature PCA')
axes[0].set_ylabel('Diferenca absoluta de medias')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(diff.median(), color='navy', linestyle='--', alpha=0.6,
                label=f'Mediana = {diff.median():.3f}')
axes[0].legend()

# --- Heatmap de correlacao entre features PCA ---
# PCA produz componentes descorrelacionados por construcao
# Esperamos valores proximos de 0 fora da diagonal
# Se houver correlacoes altas, o dataset pode ter sofrido transformacoes adicionais
corr_matrix = df[pca_cols[:14]].corr()
# mask=triu oculta o triangulo superior (valores simetricos e redundantes)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[1],
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
axes[1].set_title('Correlacao entre Features PCA (V1-V14)\nEsperamos valores proximos de 0', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao das top 6 features mais discriminativas
top_features = diff.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribuicao das 6 Features PCA Mais Discriminativas', fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(top_features):
    # clip() remove outliers extremos para melhor visualizacao
    # usamos percentis 1% e 99% como limites (remove 2% dos valores extremos)
    p1, p99 = df[col].quantile([0.01, 0.99])
    legit_vals = legit[col].clip(p1, p99)
    fraud_vals = fraud[col].clip(p1, p99)

    axes[i].hist(legit_vals, bins=80, density=True, alpha=0.5, color='#27ae60', label='Legitima')
    axes[i].hist(fraud_vals, bins=40, density=True, alpha=0.7, color='#e74c3c', label='Fraude')

    # Linhas verticais nas medias evidenciam o deslocamento entre classes
    # Quanto maior o deslocamento, mais discriminativa a feature
    axes[i].axvline(legit[col].mean(), color='#1a8a4a', linestyle='--', linewidth=2,
                    label=f'Media legit: {legit[col].mean():.2f}')
    axes[i].axvline(fraud[col].mean(), color='#c0392b', linestyle='--', linewidth=2,
                    label=f'Media fraud: {fraud[col].mean():.2f}')

    axes[i].set_title(f'{col} — delta medias = {diff[col]:.3f}', fontweight='bold')
    axes[i].set_xlabel('Valor da feature')
    axes[i].set_ylabel('Densidade')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Distribuicoes separadas confirmam que o Autoencoder')
print('tera dificuldade em reconstruir fraudes -> alto reconstruction_error.')

---
## 6. Validação da Hipótese do Autoencoder

Antes de treinar o Autoencoder, podemos validar sua hipótese central com uma análise simples:

> Se fraudes têm padrão diferente das legítimas, elas devem estar **mais distantes do centroide legítimo** no espaço de features.

Usamos distância euclidiana normalizada como proxy do `reconstruction_error` para confirmar isso antes de qualquer treinamento.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Normaliza as features para comparacao justa entre dimensoes
# Sem normalizacao, features com maior escala dominariam a distancia
scaler_preview = StandardScaler()
X_scaled = scaler_preview.fit_transform(df[pca_cols + ['Amount', 'Time']])
y = df['Class'].values

# Centroide das transacoes legitimas
# No Autoencoder real, o decoder aprende a reconstruir esse 'centro'
# Aqui usamos a media simples como aproximacao
centroid_legit = X_scaled[y == 0].mean(axis=0)

# Distancia euclidiana media de cada transacao ao centroide
# sqrt(mean((x - centroide)^2)) — mesma formula do MSE do Autoencoder
distances = np.sqrt(((X_scaled - centroid_legit) ** 2).mean(axis=1))

ratio_dist = distances[y==1].mean() / distances[y==0].mean()
print('Distancia media ao centroide legitimo:')
print(f'  Legitimas : {distances[y==0].mean():.4f} +/- {distances[y==0].std():.4f}')
print(f'  Fraudes   : {distances[y==1].mean():.4f} +/- {distances[y==1].std():.4f}')
print(f'\nFraudes estao {ratio_dist:.2f}x mais distantes do centroide legitimo.')
print('Hipotese do Autoencoder confirmada.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Proxy do Reconstruction Error — Validacao da Hipotese do Autoencoder', fontsize=14, fontweight='bold')

# --- Histograma das distancias ---
# Visualizamos ate p99 das legitimas para nao distorcer o grafico com outliers extremos
p99_legit = np.percentile(distances[y==0], 99)

axes[0].hist(distances[y==0], bins=100, density=True, alpha=0.6,
             color='#27ae60', label='Legitima', range=(0, p99_legit*2))
axes[0].hist(distances[y==1], bins=50, density=True, alpha=0.8,
             color='#e74c3c', label='Fraude', range=(0, p99_legit*2))

# Linha do threshold = percentil 95 das legitimas
# No Autoencoder real usamos exatamente esse criterio:
# tudo acima do p95 das legitimas e considerado anomalia
threshold_preview = np.percentile(distances[y==0], 95)
axes[0].axvline(threshold_preview, color='navy', linestyle='--', linewidth=2,
                label=f'Threshold p95 = {threshold_preview:.3f}')
axes[0].set_title('Distribuicao das Distancias por Classe', fontweight='bold')
axes[0].set_xlabel('Distancia ao centroide legitimo')
axes[0].set_ylabel('Densidade')
axes[0].legend()

# --- Boxplot comparativo ---
bp = axes[1].boxplot([distances[y==0], distances[y==1]],
                      labels=['Legitima', 'Fraude'],
                      patch_artist=True,
                      medianprops=dict(color='#2c3e50', linewidth=2.5),
                      flierprops=dict(marker='o', markersize=2, alpha=0.2))
bp['boxes'][0].set_facecolor('#27ae6055')
bp['boxes'][1].set_facecolor('#e74c3c55')
axes[1].set_title('Boxplot das Distancias por Classe', fontweight='bold')
axes[1].set_ylabel('Distancia')
axes[1].set_ylim(0, np.percentile(distances, 99))

plt.tight_layout()
plt.show()

---
## 7. Resumo e Decisões de Modelagem

Esta seção consolida tudo que aprendemos nas seções anteriores em **decisões concretas de modelagem**.

Uma EDA bem feita não termina em gráficos — termina em **justificativas documentadas** para cada escolha técnica. Recrutadores e revisores de código querem ver que você não apenas explorou os dados, mas que extraiu **insights acionáveis** deles.

### O que a EDA nos ensinou e como isso impacta o modelo:

| Observação na EDA | Decisão de modelagem | Impacto |
|---|---|---|
| 0,17% de fraudes | PR-AUC como métrica principal | AUC-ROC seria enganoso — 99,83% de acurácia com modelo inútil |
| Amount e Time fora de escala | StandardScaler só em Amount e Time | Autoencoder (MSE) penalizaria desproporcionalmente features com maior magnitude |
| V1–V28 já normalizadas via PCA | Sem transformação extra nas PCA features | Evita dupla normalização que distorceria as componentes |
| Fraudes X vezes mais distantes do centróide | Autoencoder treinado só com legítimas | Alta separabilidade no espaço de features confirma a hipótese do anomaly detection |
| 0,17% extremo para supervisionado | SMOTE com `sampling_strategy=0.1` | Eleva fraudes para ~10% sem ir a 50/50, preservando realismo |
| Threshold 0,5 não faz sentido | Threshold calibrado pela curva PR | Maximiza F1 no ponto ótimo real, não em um valor arbitrário |

> **Nota sobre o pipeline híbrido:** a combinação Autoencoder + XGBoost não é acaso — cada componente resolve uma limitação do outro. O Autoencoder detecta anomalias sem depender de labels balanceados; o XGBoost faz a decisão final supervisionada com o `reconstruction_error` como feature extra, unindo os dois mundos.

In [ ]:
ratio_dist = distances[y==1].mean() / distances[y==0].mean()

print('=' * 62)
print('  RESUMO DA EDA — DECISOES DE MODELAGEM')
print('=' * 62)
print(f'''
DATASET
  Transacoes totais  : {len(df):,}
  Fraudes            : {class_counts[1]:,} ({fraud_pct:.4f}%)
  Valores nulos      : Nenhum
  Tipos de dado      : Todos float64

DECISOES BASEADAS NA EDA

  1. METRICA PRINCIPAL -> PR-AUC (nao Acuracia, nao AUC-ROC)
     0.17% de fraudes torna Acuracia e AUC-ROC enganosos.
     PR-AUC mede performance exatamente na classe minoritaria.

  2. PRE-PROCESSAMENTO -> StandardScaler apenas em Amount e Time
     V1-V28 ja normalizados pelo PCA.
     Amount e Time tem escalas incompativeis com as features PCA.

  3. AUTOENCODER -> treinado apenas com legitimas
     Fraudes ficam {ratio_dist:.1f}x mais distantes do padrao legitimo.
     Threshold = percentil 95 das legitimas no conjunto de teste.

  4. SMOTE -> sampling_strategy=0.1 (fraudes elevadas para 10%%)
     0.17%% e extremo demais para classificadores supervisionados.
     Nao vamos a 50/50 para preservar realismo na distribuicao.

  5. THRESHOLD CALIBRADO -> maximiza F1 na curva Precision-Recall
     Threshold 0.5 e arbitrario para dados desbalanceados.
     O ponto otimo e encontrado na curva PR apos o treino.
''')
print('=' * 62)
print()
print('Proximos passos:')
print('  python src/data/make_dataset.py')
print('  python src/models/train_autoencoder.py')
print('  python src/models/train_classifier.py')
print('  -> 02_model_evaluation.ipynb')

---
## ✅ EDA Concluída — Próximos Passos

A análise exploratória está completa. Todas as decisões de modelagem foram **justificadas pelos dados**, não por convenção.

### Pipeline de execução

Execute os scripts na ordem abaixo antes de abrir o notebook de avaliação:

```bash
# 1. Preparar dados (split treino/teste + scaler)
python src/data/make_dataset.py

# 2. Treinar o Autoencoder (anomaly detection — só legítimas)
python src/models/train_autoencoder.py

# 3. Treinar o XGBoost (supervisionado + SMOTE + reconstruction_error)
python src/models/train_classifier.py
```

### O que vem a seguir

**`02_model_evaluation.ipynb`** cobre:

| Seção | Conteúdo |
|---|---|
| Curva Precision-Recall | Comparação visual entre modelos — por que PR-AUC e não ROC |
| Matriz de confusão | Análise de FP e FN com custo de negócio associado |
| Comparação de modelos | Logistic Regression vs Random Forest vs XGBoost vs Autoencoder+XGBoost |
| Calibração do threshold | Como o ponto ótimo foi encontrado na curva PR |
| Feature importance | Quais features PCA mais contribuem — e o papel do `reconstruction_error` |
| Análise de erros | Quais fraudes o modelo ainda erra e por quê |

---

*Dataset: [Credit Card Fraud Detection — ULB / Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)*